# Matmul基础API

## 概述

上一节我们学习了Cube计算的存储层级、分形格式和四个基础API（`asc.data_copy`、`asc.load_data`、`asc.mmad`、`asc.fixpipe`）。本节通过代码片段逐一展示基础API在使用上的痛点，帮助你理解为什么需要高阶API的封装。

### 学习目标

1. 理解Matmul的计算原理和维度关系
2. 掌握Cube存储层级（GM→L1→L0A/L0B→L0C→GM）的数据流路径
3. 理解分形格式（ZZ/ZN/NZ）对数据排布的要求
4. 体会基础API在Matmul开发中的痛点，理解高阶API的价值

---
# 1. Matmul计算原理

Matmul计算 **C = A × B**，其中A是 `[M, K]` 的左矩阵，B是 `[K, N]` 的右矩阵，C是 `[M, N]` 的结果矩阵。

```
        K             N
    ┌─────────┐   ┌─────────┐
  M │   A     │ × │   B     │ K = C [M, N]
    └─────────┘   └─────────┘
```

每个元素 `C[i,j] = Σ(k=0..K-1) A[i,k] × B[k,j]`

### 与Vector算子的关键区别

| 维度 | Vector算子（Add/ReLU等） | Cube算子（Matmul） |
| --- | --- | --- |
| 计算单元 | Vector单元 | Cube单元 |
| 存储层级 | GM ↔ UB（两级） | GM → L1 → L0A/L0B → L0C → GM（四级） |
| 数据格式 | ND格式 | ND/ZZ/ZN/NZ格式 |
| API层级 | `asc.add`/`asc.leaky_relu` | `asc.data_copy`/`asc.load_data`/`asc.mmad`/`asc.fixpipe` |

Cube算子的数据流比Vector算子复杂得多，理解存储层级是开发Matmul算子的第一步。

---
# 2. Cube存储层级

昇腾NPU的Cube计算单元使用**四级存储层级**，数据需要逐级搬运才能参与计算：

```
Global Memory (GM)            ← ND格式
    │
    ▼ asc.data_copy (GM→L1)    ← ND→NZ 格式转换
L1 (A1/B1)                     ← NZ分形格式
    │
    ▼ asc.load_data (L1→L0A/L0B)
L0A / L0B (A2/B2)              ← Cube输入缓冲区
    │
    ▼ asc.mmad
L0C (CO1)                      ← Cube结果缓冲区
    │
    ▼ asc.fixpipe (CO1→GM)
Global Memory (GM)             ← ND格式
```

### 各级存储职责

| 存储位置 | TPosition | 职责 | 数据格式 |
| --- | --- | --- | --- |
| **GM** | `TPosition.GM` | 全局内存，存放输入A/B和输出C | ND格式 |
| **L1 (A1)** | `TPosition.A1` | 左矩阵A的L1缓冲区 | NZ分形格式 |
| **L1 (B1)** | `TPosition.B1` | 右矩阵B的L1缓冲区 | NZ分形格式 |
| **L0A (A2)** | `TPosition.A2` | 左矩阵A的Cube输入缓冲区，mmad直接从此读取 | ZZ分形格式 |
| **L0B (B2)** | `TPosition.B2` | 右矩阵B的Cube输入缓冲区，mmad直接从此读取 | ZN分形格式 |
| **L0C (CO1)** | `TPosition.CO1` | mmad计算结果缓冲区 | NZ分形格式 |

### 数据搬运通路

Cube数据搬运使用三个不同的底层指令，各司其职：

| 通路 | 指令 | 源 → 目的 | 格式转换 |
| --- | --- | --- | --- |
| GM → L1 | `asc.data_copy` + `Nd2NzParams` | GlobalTensor → LocalTensor(A1/B1) | ND → NZ |
| L1 → L0A/L0B | `asc.load_data` | LocalTensor(A1/B1) → LocalTensor(A2/B2) | NZ → ZZ/ZN |
| L0C → GM | `asc.fixpipe` | LocalTensor(CO1) → GlobalTensor | NZ → ND |

> **关键约束**：
> - GM中的数据是ND格式，搬运到L1时通过 `asc.data_copy` 传入 `asc.Nd2NzParams` 参数完成 ND→NZ 格式转换。
> - `asc.mmad`的左矩阵必须位于A2，右矩阵必须位于B2，结果必须输出到CO1。
> - mmad要求ABC矩阵的排布格式分别为ZZ、ZN、NZ。L1→L0A/L0B的 `load_data` 负责将NZ格式数据加载到mmad所需的输入缓冲区。

<img src="./images/04.02_matmul_basics_and_low_level_api/cube_memory_hierarchy.png" alt="Cube存储层级" width="500px">

---
# 3. 分形格式

Cube计算单元不直接处理ND格式的矩阵，而是要求输入数据按特定**分形格式**排布。分形是Cube硬件的基本计算单元，大小由数据类型决定：

| 数据类型 | 分形大小 | 说明 |
| --- | --- | --- |
| fp16/bf16 | 16×16 | 每个分形含256个元素 |
| fp32 | 16×8 (A) / 8×16 (B) | 每个分形含128个元素 |

### 三种分形排布

在MMAD指令中，ABC矩阵的排布格式分别为ZZ、ZN、NZ。

下图以half类型、32×32矩阵为例，展示ND格式到NZ分形格式的转换过程。32×32矩阵在N方向和D方向各划分为2个16×16分形，共4个分形。ND格式在GM中按行主序连续存储，搬运到L1后重排为NZ分形格式——每个分形内部按行主序存储，分形之间按列主序排列（即F(d₀,n₀)→F(d₁,n₀)→F(d₀,n₁)→F(d₁,n₁)，先沿D轴变化，再沿N轴）。

<img src="./images/04.02_matmul_basics_and_low_level_api/fractal_format_nd2nz.png" alt="ND到NZ分形格式转换" width="800px">


---
# 4. 基础API介绍

实现Matmul需要四个基础API协同工作，分别负责不同层级间的数据搬运和计算：

### 4.1 asc.data_copy — GM到L1搬运（ND→NZ格式转换）

`asc.data_copy`（DataCopy指令）负责将数据从GM搬运到L1(A1/B1)，**同时在搬运过程中完成ND格式到NZ分形格式的转换**。通过传入 `asc.Nd2NzParams` 参数控制格式转换：

```python
# GM → L1：ND格式 → NZ格式
# Nd2NzParams(nd_num, n_value, d_value, src_nd_matrix_stride, src_d_value,
#             dst_nz_c0_stride, dst_nz_n_stride, dst_nz_matrix_stride)
nd2nz_params = asc.Nd2NzParams(
    1,     # nd_num: ND矩阵个数
    16,    # n_value: N维度大小
    16,    # d_value: D维度大小
    0,     # src_nd_matrix_stride: 源ND矩阵间步长
    16,    # src_d_value: 源D维度实际值
    16,    # dst_nz_c0_stride: 目的NZ的C0步长
    1,     # dst_nz_n_stride: 目的NZ的N方向步长
    0      # dst_nz_matrix_stride: 目的NZ矩阵间步长
)
asc.data_copy(a_l1, a_gm, intri_params=nd2nz_params)   # GM → A1 (ND→NZ)
asc.data_copy(b_l1, b_gm, intri_params=nd2nz_params)   # GM → B1 (ND→NZ)
```

**Nd2NzParams 参数说明**：

| 参数 | 类型 | 说明 |
| --- | --- | --- |
| `nd_num` | int | ND矩阵个数，通常为1 |
| `n_value` | int | ND矩阵的N维度大小（列数） |
| `d_value` | int | ND矩阵的D维度大小（行数） |
| `src_nd_matrix_stride` | int | 源操作数中相邻ND矩阵间的步长 |
| `src_d_value` | int | 源操作数中D维度的实际值 |
| `dst_nz_c0_stride` | int | 目的NZ格式中C0方向的步长（fp16下C0=16，可通过 `asc.property(asc.DEFAULT_C0_SIZE)` 获取） |
| `dst_nz_n_stride` | int | 目的NZ格式中N方向的步长 |
| `dst_nz_matrix_stride` | int | 目的操作数中相邻NZ矩阵间的步长 |

### 4.2 asc.load_data — L1到L0A/L0B搬运

`asc.load_data`（LoadData指令）负责Cube内部的L1→L0A/L0B搬运，将分形格式数据加载到mmad可直接读取的输入缓冲区：

```python
# L1 → L0A/L0B：源为LocalTensor(A1/B1)，目的为LocalTensor(A2/B2)
asc.load_data(a_l0a, a_l1, params)  # A1 → A2
asc.load_data(b_l0b, b_l1, params)  # B1 → B2
```

**LoadData2DParams 参数说明**：

| 参数 | 类型 | 说明 |
| --- | --- | --- |
| `start_index` | int | 起始分形ID |
| `repeat_times` | int | 迭代次数|
| `src_stride` | int | 源相邻迭代间隔 |
| `sid` | int | 预留参数，设为0 |
| `dst_gap` | int | 目的相邻迭代间隔 |
| `if_transpose` | bool | 是否转置 |
| `addr_mode` | int | 预留参数，设为0 |

### 4.3 asc.mmad — 矩阵乘加

执行 `C += A × B`，A在L0A(A2)，B在L0B(B2)，结果在L0C(CO1)。

```python
asc.mmad(dst_c, fm_a, filter_b, params)
```

**MmadParams 参数说明**：

| 参数 | 类型 | 说明 |
| --- | --- | --- |
| `m` | int | 左矩阵高度（M维度），取值范围[0, 4095] |
| `n` | int | 右矩阵宽度（N维度），取值范围[0, 4095] |
| `k` | int | 左矩阵宽/右矩阵高（K维度），取值范围[0, 4095] |
| `cmatrixInitVal` | bool | C矩阵初始值是否为0，默认True。False时执行累加 `C += A×B` |

### 4.4 asc.fixpipe — 结果搬出（NZ→ND格式转换）

将CO1中的NZ格式结果搬到GM恢复为ND格式，可执行量化、格式转换等后处理。

```python
asc.fixpipe(dst_gm, src_c, params)
```

**FixpipeParamsV220 参数说明**：

| 参数 | 类型 | 说明 |
| --- | --- | --- |
| `n_size` | int | 源NZ矩阵在N方向上的大小 |
| `m_size` | int | 源NZ矩阵在M方向上的大小 |
| `src_stride` | int | 源NZ矩阵中相邻Z排布的起始地址偏移，单位C0_Size |
| `dst_stride` | int | 目的矩阵的地址步长，单位datablock(32B) |
| `quant_pre` | QuantModes | 量化模式，默认 `asc.QuantModes.NoQuant`（不使能量化） |
| `deq_scalar` | int | scalar量化参数，仅在quantPre为scalar量化时设置 |
| `nd_num` | int | 源NZ矩阵的数目，默认1 |
| `src_nd_stride` | int | 不同NZ矩阵起始地址间隔，nd_num=1时设为0 |
| `dst_nd_stride` | int | 目的相邻ND矩阵起始地址偏移，nd_num=1时设为0 |
| `relu_en` | bool | 是否使能ReLU，默认False |
| `unit_flag` | int | Mmad与Fixpipe指令细粒度并行控制，0为保留值 |
| `is_channel_split` | bool | 是否使能通道拆分，默认False |

### 4.5 asc.set_flag / asc.wait_flag — 流水线同步

Cube计算的四个API分别运行在不同的硬件流水线上，当相邻两步存在数据依赖时，必须插入 `set_flag`/`wait_flag` 同步指令，确保前一步完成后才开始下一步。

**各API对应的流水线**：

| API | 流水线 | PipeID | 说明 |
| --- | --- | --- | --- |
| `asc.data_copy` (GM→L1) | MTE2 | `PIPE_MTE2` | 外部内存到L1的搬运 |
| `asc.load_data` (L1→L0A/L0B) | MTE1 | `PIPE_MTE1` | L1到L0的搬运 |
| `asc.mmad` | M | `PIPE_M` | 矩阵乘加计算 |
| `asc.fixpipe` (CO1→GM) | FIX | `PIPE_FIX` | 结果搬出 |

**HardEvent 同步事件说明**：

| HardEvent | 含义 | 使用场景 |
| --- | --- | --- |
| `MTE2_MTE1` | MTE2 → MTE1 | data_copy 完成后、load_data 开始前 |
| `MTE1_M` | MTE1 → M | load_data 完成后、mmad 开始前 |
| `M_FIX` | M → FIX | mmad 完成后、fixpipe 开始前 |

**同步指令用法**：

```python
# set_flag/wait_flag 必须成对出现，event_id 通过 fetch_event_id 获取
pipe = asc.TPipe()

# MTE2 → MTE1 同步：data_copy 完成后通知 load_data
event_id = pipe.fetch_event_id(event=asc.HardEvent.MTE2_MTE1)
asc.set_flag(event=asc.HardEvent.MTE2_MTE1, event_id=event_id)
asc.wait_flag(event=asc.HardEvent.MTE2_MTE1, event_id=event_id)

# MTE1 → M 同步：load_data 完成后通知 mmad
event_id = pipe.fetch_event_id(event=asc.HardEvent.MTE1_M)
asc.set_flag(event=asc.HardEvent.MTE1_M, event_id=event_id)
asc.wait_flag(event=asc.HardEvent.MTE1_M, event_id=event_id)

# M → FIX 同步：mmad 完成后通知 fixpipe
event_id = pipe.fetch_event_id(event=asc.HardEvent.M_FIX)
asc.set_flag(event=asc.HardEvent.M_FIX, event_id=event_id)
asc.wait_flag(event=asc.HardEvent.M_FIX, event_id=event_id)
```

> **关键约束**：
> - `set_flag` 和 `wait_flag` 必须成对出现，缺一不可。
> - `event_id` 必须通过 `pipe.fetch_event_id()` 获取，禁止自行指定，否则会与框架同步事件冲突导致死锁。
> - 同一流水线内部有数据依赖时使用 `asc.pipe_barrier()`，不同流水线之间使用 `set_flag`/`wait_flag`。

下图展示了四个基础API各自运行的流水线，以及三组同步指令的插入位置。注意每个同步指令都跨越相邻两条流水线，确保数据依赖正确。

<img src="./images/04.02_matmul_basics_and_low_level_api/pipeline_sync.png" alt="Cube计算流水线与同步机制" width="700px">

---
# 5. 基础API的痛点

如果用四个基础API从零实现一个Matmul，开发者需要手动处理大量底层细节。下面逐一列举基础API的六大痛点，每个痛点配一段代码片段说明。

### 痛点1：手动管理五级存储的Tensor分配和地址

Cube算子数据流经 GM→L1→L0A/L0B→L0C→GM 五级存储，每级都需要手动创建 `LocalTensor` 并指定存储位置（`TPosition`）、起始地址（`addr`）和分片大小（`tile_size`）。地址分配不当会导致缓冲区冲突，且难以调试。

```python
# 左矩阵A: L1、L0A 各需一个Tensor，地址和大小都要手动算
a_l1  = asc.LocalTensor(dtype=asc.float16, pos=asc.TPosition.A1,  addr=0,               tile_size=M*K)
a_l0a = asc.LocalTensor(dtype=asc.float16, pos=asc.TPosition.A2,  addr=0,               tile_size=M*K)
# 右矩阵B: L1的addr要跳过A占用的空间，L0B另起地址
b_l1  = asc.LocalTensor(dtype=asc.float16, pos=asc.TPosition.B1,  addr=M*K*2,           tile_size=K*N)
b_l0b = asc.LocalTensor(dtype=asc.float16, pos=asc.TPosition.B2,  addr=0,               tile_size=K*N)
# 结果C: CO1
c_l0c = asc.LocalTensor(dtype=asc.float32, pos=asc.TPosition.CO1, addr=0,               tile_size=M*N)
```

对比Vector算子只需 GM↔UB 两级存储，Cube算子的存储管理复杂度显著更高。

### 痛点2：区分 data_copy 和 load_data 两个搬运指令

同样是数据搬运，GM→L1 用 `asc.data_copy`，L1→L0A/L0B 用 `asc.load_data`，两者参数体系完全不同，开发者必须牢记每段路径对应哪个指令：

```python
# GM → L1: data_copy + Nd2NzParams (8个参数)
asc.data_copy(a_l1, a_gm, intri_params=nd2nz_a)

# L1 → L0A/L0B: load_data + LoadData2DParams (7个参数)
asc.load_data(a_l0a, a_l1, load_params)
asc.load_data(b_l0b, b_l1, load_params)
```

### 痛点3：手动计算 Nd2NzParams 的 8 个参数

`Nd2NzParams` 描述ND→NZ格式转换，共有8个参数，需要根据矩阵形状和分形大小手动计算步长，极易出错：

```python
# Nd2NzParams(nd_num, n_value, d_value, src_nd_matrix_stride, src_d_value,
#             dst_nz_c0_stride, dst_nz_n_stride, dst_nz_matrix_stride)
nd2nz_a = asc.Nd2NzParams(
    1,              # nd_num
    M,              # n_value
    K,              # d_value
    0,              # src_nd_matrix_stride
    K,              # src_d_value
    M // 16,        # dst_nz_c0_stride  ← 需根据C0=16计算
    1,              # dst_nz_n_stride
    0               # dst_nz_matrix_stride
)
```

### 痛点4：手动插入 set_flag/wait_flag 流水同步

四个API分属MTE2/MTE1/M/FIX四条流水线，相邻两步之间必须成对插入 `set_flag`/`wait_flag` 同步指令，`event_id` 还要通过 `pipe.fetch_event_id()` 获取。一个完整Matmul至少需要3组同步，K方向分块时更多：

```python
# 每一步搬运/计算之间都要重复这段同步模板
event_id = pipe.fetch_event_id(event=asc.HardEvent.MTE2_MTE1)
asc.set_flag(event=asc.HardEvent.MTE2_MTE1, event_id=event_id)
asc.wait_flag(event=asc.HardEvent.MTE2_MTE1, event_id=event_id)
# ... load_data ...
event_id = pipe.fetch_event_id(event=asc.HardEvent.MTE1_M)
asc.set_flag(event=asc.HardEvent.MTE1_M, event_id=event_id)
asc.wait_flag(event=asc.HardEvent.MTE1_M, event_id=event_id)
# ... mmad ...
event_id = pipe.fetch_event_id(event=asc.HardEvent.M_FIX)
asc.set_flag(event=asc.HardEvent.M_FIX, event_id=event_id)
asc.wait_flag(event=asc.HardEvent.M_FIX, event_id=event_id)
```

漏写或写错同步指令会导致数据竞争或死锁，且难以定位。

### 痛点5：K方向分块循环与累加控制

当K维度较大时，需要将K轴切分为多个块，逐块搬运并累加到L0C。开发者要手动编写循环、计算K块偏移，并通过 `cmatrix_init_val` 控制首次清零/后续累加，还要在迭代间插入 `MTE1→MTE2` 同步：

```python
for i in range(K_ITER):
    # 手动计算第i个K块的GM偏移
    asc.data_copy(a_l1, a_gm[i * K_BLOCK], intri_params=nd2nz_a)
    asc.data_copy(b_l1, b_gm[i * K_BLOCK * N], intri_params=nd2nz_b)
    # ... 同步 + load_data + 同步 ...
    # 首次迭代清零，后续迭代累加
    mmad_params = asc.MmadParams(m=M, n=N, k=K_BLOCK,
                                 cmatrix_init_val=(i == 0))
    asc.mmad(c_l0c, a_l0a, b_l0b, mmad_params)
    # 迭代间同步：确保本次load_data完成后才允许下次data_copy覆写L1
    if i < K_ITER - 1:
        event_id = pipe.fetch_event_id(event=asc.HardEvent.MTE1_MTE2)
        asc.set_flag(event=asc.HardEvent.MTE1_MTE2, event_id=event_id)
        asc.wait_flag(event=asc.HardEvent.MTE1_MTE2, event_id=event_id)
```

### 痛点6：手动计算 stride / tile_size 等硬件参数

`fixpipe` 的 `dst_stride` 单位是 datablock(32B)，`tile_size` 要按元素个数算，Nd2NzParams 的步长要按分形大小算……这些硬件相关参数分散在各处，缺乏统一抽象：

```python
# dst_stride单位是datablock(32B)，需手动换算
fixpipe_params = asc.FixpipeParamsV220(
    n_size=N, m_size=M, src_stride=0,
    dst_stride=N * 4 // 32,   # ← 32B换算
    quant_pre=asc.QuantModes.NoQuant, deq_scalar=0,
    nd_num=1, src_nd_stride=0, dst_nd_stride=0,
    relu_en=False, unit_flag=0, is_channel_split=False
)
```

---

### 小结

| 痛点 | 核心问题 |
| --- | --- |
| 1. 五级存储管理 | 手动分配Tensor地址和tile_size，易冲突 |
| 2. 两个搬运指令 | data_copy/load_data 参数体系不同，需区分 |
| 3. Nd2NzParams | 8个参数手动计算，易出错 |
| 4. 流水同步 | 每步插入set_flag/wait_flag，漏写致死锁 |
| 5. K方向分块 | 手动循环、偏移、累加控制、迭代间同步 |
| 6. 硬件参数 | stride/tile_size单位换算分散 |

> 以上六大痛点正是高阶API `asc.adv.Matmul` 要解决的问题——它封装了存储管理、格式转换、流水同步、K方向分块等全部细节，让Matmul开发变得简洁高效。下一节我们将学习高阶API。

---
# 6. 课后练习

### 选择题

**1.** Cube计算的数据流路径是？

- A. GM → UB → L0A/L0B → L0C → GM
- B. GM → L1(A1/B1) → L0A/L0B(A2/B2) → mmad → L0C(CO1) → fixpipe → GM
- C. GM → L0C → L0A/L0B → GM
- D. GM → UB → GM

**2.** 在fp16数据类型下，Cube矩阵的分形大小是多少？

- A. 8×8
- B. 16×16
- C. 32×32
- D. 16×32

### 填空题

**3.** `asc.mmad` 的左矩阵（fm参数）必须位于哪个存储位置？（写出TPosition枚举值）

**4.** GM到L1的数据搬运使用 ______ 指令并传入 ______ 参数，该过程将GM中的 ______ 格式数据转换为L1上的 ______ 格式。

**5.** data_copy（MTE2流水线）完成后、load_data（MTE1流水线）开始前，需要插入 ______ 和 ______ 同步指令，对应的HardEvent为 ______ 。

**6.** `asc.mmad` 的 `MmadParams` 中，`m`、`n`、`k` 三个参数分别对应矩阵乘法 `C[M,N] = A[M,K] × B[K,N]` 中的哪个维度？当K维度较大需要分块累加时，`cmatrix_init_val` 参数在首次迭代应设为 ______ ，后续迭代应设为 ______ 。

---

> 点击下方查看答案

In [ ]:
!cat ./answer/04.02_matmul_basics_and_low_level_api/practice_answers.md